# Week 7 Lab — Time series with pandas

**HWRS 564a · Fall 2026**

Almost everything in hydrology is a time series, and pandas has a whole second
personality for them. Give a DataFrame a `DatetimeIndex` and you can ask for
"2015", "every August", "a 30-day average", or "annual totals by water year"
without writing a single loop over dates.

This week is that personality: `resample`, `groupby`, `rolling`, and reshaping
between long and wide.

## How to use this notebook

Run each cell with **Shift+Enter**. Cells marked  **`# YOUR TURN`**  have
something for you to write. Cells marked **`# CHECK`** verify your answer — if
they run without complaint, you're right.

> **Before you submit anything all semester:** *Kernel → Restart Kernel and Run
> All Cells*. A notebook that only works when run out of order is not finished.


## Learning objectives

By the end of this notebook you can:

1. Build a `DatetimeIndex` and select periods by partial string
2. Use the `.dt` accessor to pull year, month, and day-of-year out of a column
3. `resample` a daily series to monthly and annual, and pick the right aggregation
4. Group by a derived key — including the **water year**, which is not the calendar year
5. Smooth with `rolling`, and say what a centred window does to the ends
6. Reshape long to wide with `pivot_table`, and back with `melt`

---

## Part 1 — The DatetimeIndex

Reading dates as text gets you strings. Reading them as dates gets you a whole
API.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

ROOT = next(p for p in Path.cwd().parents if (p / "pyproject.toml").exists())
DATA = ROOT / "data"

flow = pd.read_csv(DATA / "cache" / "nwis_09484000_dv.csv", parse_dates=["datetime"])
flow = flow.set_index("datetime").sort_index()

print(type(flow.index))
print(f"{len(flow):,} days, {flow.index.min():%Y-%m-%d} to {flow.index.max():%Y-%m-%d}")
flow.head(3)

With a `DatetimeIndex` you can slice with **partial strings**. This is the
single most useful thing in this notebook.

In [ ]:
print(f'flow.loc["2006"]         -> {len(flow.loc["2006"]):,} rows (a whole year)')
print(f'flow.loc["2006-08"]      -> {len(flow.loc["2006-08"]):,} rows (one month)')
print(f'flow.loc["2006-07":"2006-09"] -> {len(flow.loc["2006-07":"2006-09"]):,} rows')

Note that a `.loc` slice on dates is **inclusive at both ends** — the same rule
as label slices from Week 5, and here it is what you want.

The `.dt` accessor does the same job on a *column* that isn't the index.

In [ ]:
levels = pd.read_csv(DATA / "tucson_water_levels.csv",
                     dtype={"site_no": str}, parse_dates=["date"])

print(levels["date"].dt.year.head(3).tolist())
print(levels["date"].dt.month.head(3).tolist())
print(levels["date"].dt.dayofyear.head(3).tolist())
print(levels["date"].dt.quarter.head(3).tolist())

### YOUR TURN 1

Pull out the monsoon of 2006 — **July through September** — and describe it.

- `monsoon_2006` — the rows of `flow` for those three months
- `monsoon_peak` — the largest daily discharge in that window
- `monsoon_days` — how many days are in it

In [ ]:
# YOUR TURN
monsoon_2006 = ...
monsoon_peak = ...
monsoon_days = ...

In [ ]:
# CHECK
assert monsoon_days == 92, f"July-September is 92 days, got {monsoon_days}"
assert isinstance(monsoon_2006.index, pd.DatetimeIndex), "you lost the index somewhere"
assert monsoon_2006.index.min().month == 7 and monsoon_2006.index.max().month == 9
assert abs(monsoon_peak - 2450.0) < 1.0, f"expected about 2450 cfs, got {monsoon_peak}"
print(f"{monsoon_days} days, peaking at {monsoon_peak:,.0f} cfs")
print("Correct.")

---

## Part 2 — Resampling

`resample` is `groupby` for time. You give it a frequency and an aggregation.

In [ ]:
q = flow["discharge_cfs"]

monthly = q.resample("ME").mean()      # ME = month end
annual = q.resample("YE").mean()       # YE = year end

print(f"{len(q):,} daily -> {len(monthly)} monthly -> {len(annual)} annual")
print()
print(annual.round(1).head(5).to_string())

**Choosing the aggregation is a hydrology decision, not a pandas one.**

| You want | Use |
|---|---|
| average conditions | `.mean()` |
| total volume of water | `.sum()` |
| flood risk | `.max()` |
| a typical day, ignoring floods | `.median()` |

For discharge in cfs, `.sum()` over a month gives you *cfs-days* — a volume, and
a perfectly good one, as long as you say so in the axis label.

In [ ]:
print(f"wettest year: {annual.idxmax():%Y} at {annual.max():.1f} cfs mean daily")
print(f"driest year:  {annual.idxmin():%Y} at {annual.min():.1f} cfs mean daily")
print(f"ratio: {annual.max() / annual.min():.0f}x between the two")

A factor of ten between the wettest and driest year, in the same creek. Any
analysis that uses "the average year" is describing a year that does not occur.

### The water year

Hydrologists count years from **October 1 to September 30**, so that a winter
snowpack and the melt it produces land in the same year. Calendar years split
them, which makes every winter storm look like it belongs to the wrong season.

Pandas knows about this: `resample("YE-SEP")` ends years in September.

In [ ]:
calendar_totals = q.resample("YE").sum()
water_year_totals = q.resample("YE-SEP").sum()

print(f"calendar years: {len(calendar_totals)}")
print(f"water years:    {len(water_year_totals)}")
print()
print("water-year totals, cfs-days:")
print(water_year_totals.head(4).round(0).to_string())

### YOUR TURN 2

Build the **monthly climatology** of Sabino Creek: the long-term average flow in
each calendar month, across all twenty years.

This is not `resample` — resample would give you 240 separate months. You want
all the Januaries averaged together, so group by the month *number*.

In [ ]:
# YOUR TURN
climatology = ...

In [ ]:
# CHECK
assert len(climatology) == 12, f"expected 12 values, got {len(climatology)}"
assert climatology.index.min() == 1 and climatology.index.max() == 12
assert climatology.idxmax() == 1, f"expected January to be wettest, got {climatology.idxmax()}"
assert climatology.idxmin() == 6, f"expected June to be driest, got {climatology.idxmin()}"
print(climatology.round(1).to_string())
print("\nCorrect.")

**Two peaks, and they have different causes.** January is winter frontal
storms — long, gentle, basin-wide. July and August are monsoon
thunderstorms — short, violent, local. June is the foresummer drought between
them.

A single "mean annual flow" number averages those together into a season that
does not exist. This is why the climatology plot is worth making before any
other analysis.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 3.4))
ax.bar(climatology.index, climatology.values, color="#1E5288", alpha=0.85)
ax.set_xticks(range(1, 13))
ax.set_xticklabels(["J", "F", "M", "A", "M", "J", "J", "A", "S", "O", "N", "D"])
ax.set_xlabel("month")
ax.set_ylabel("mean daily discharge (cfs)")
ax.set_title("Sabino Creek: winter storms and the monsoon, with a dry gap between")
ax.grid(alpha=0.3, axis="y")
plt.tight_layout()
plt.show()

---

## Part 3 — Rolling windows

Resampling gives you fewer, coarser points. A **rolling window** keeps every
point but averages each one with its neighbours — useful when you want to see
the shape of a season without losing the daily axis.

In [ ]:
smooth_30 = q.rolling(window=30, center=True).mean()
smooth_365 = q.rolling(window=365, center=True).mean()

print(f"original : {q.notna().sum():,} values")
print(f"30-day   : {smooth_30.notna().sum():,} values  ({smooth_30.isna().sum()} lost at the ends)")
print(f"365-day  : {smooth_365.notna().sum():,} values  ({smooth_365.isna().sum()} lost at the ends)")

**`center=True` costs you half a window at each end.** A centred 365-day mean
cannot be computed for the first or last 182 days, because half the window falls
outside the data. That is honest — the alternative, `center=False`, labels each
average with the *last* day of its window, which shifts every feature later by
half a window.

Use `center=True` when you are describing the data. Use `center=False` when you
are simulating something that could only know the past.

In [ ]:
fig, ax = plt.subplots(figsize=(11, 3.8))
ax.plot(q.index, q, color="#C9C9C9", lw=0.5, label="daily")
ax.plot(smooth_30.index, smooth_30, color="#AB0520", lw=1.2, label="30-day mean")
ax.plot(smooth_365.index, smooth_365, color="#0C234B", lw=2.2, ls="--", label="365-day mean")
ax.set_yscale("symlog", linthresh=1)
ax.set_ylabel("discharge (cfs, symlog)")
ax.set_title("Sabino Creek, 2005-2024")
ax.legend(frameon=False, ncol=3)
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

Note `symlog`. A linear axis is dominated by the 2,450 cfs peak and everything
else is a flat line at the bottom; a pure log axis cannot show the zeros at all.
`symlog` is linear near zero and logarithmic beyond — which is the right choice
for a record that spans zero to thousands.

### YOUR TURN 3

Water managers care about sustained flow, not single-day spikes. Find the
**wettest 30-day period** in the record.

- `rolling_30` — the centred 30-day rolling mean
- `wettest_window_cfs` — its maximum value
- `wettest_window_end` — the date at which that maximum sits

In [ ]:
# YOUR TURN
rolling_30 = ...
wettest_window_cfs = ...
wettest_window_end = ...

In [ ]:
# CHECK
assert abs(wettest_window_cfs - 258.2) < 1.0, f"expected about 258 cfs, got {wettest_window_cfs}"
assert wettest_window_end.year == 2006, f"expected 2006, got {wettest_window_end.year}"
print(f"wettest 30 days averaged {wettest_window_cfs:.1f} cfs, "
      f"centred on {wettest_window_end:%Y-%m-%d}")
print(f"the single wettest day was {q.max():,.0f} cfs — {q.max() / wettest_window_cfs:.0f}x higher")
print("Correct.")

The peak day is roughly ten times the wettest sustained month. That gap is the
whole difference between flood hydrology and water-supply hydrology, and it is
why the two fields argue about which statistic matters.

---

## Part 4 — Grouping by something you computed

`groupby` takes any key, including one you derive on the spot. That is how you
answer questions the index alone can't.

In [ ]:
# One row per well per year — 80 wells, ~28 years each
annual_levels = (
    levels
    .assign(year=levels["date"].dt.year)
    .groupby(["site_no", "year"])["depth_to_water_ft"]
    .mean()
    .reset_index()
)

print(f"{len(annual_levels):,} well-years from {len(levels):,} measurements")
annual_levels.head()

`.agg()` runs several aggregations at once, which saves three passes over the
data and reads better:

In [ ]:
by_site = levels.groupby("site_no")["depth_to_water_ft"].agg(
    n="size", first="first", last="last", deepest="max", shallowest="min"
)
by_site["change_ft"] = by_site["last"] - by_site["first"]

print(by_site.sort_values("change_ft", ascending=False).head(5).round(1).to_string())

### YOUR TURN 4

How widespread is the decline? Using `by_site`:

- `n_declining` — how many wells ended deeper than they started
- `n_recovering` — how many ended shallower
- `median_change` — the median change in feet across all 80 wells

In [ ]:
# YOUR TURN
n_declining = ...
n_recovering = ...
median_change = ...

In [ ]:
# CHECK
assert n_declining + n_recovering <= len(by_site)
assert n_declining == 73, f"expected 73 declining, got {n_declining}"
assert abs(median_change - 46.8) < 0.5, f"expected about 46.8 ft, got {median_change}"
print(f"{n_declining} of {len(by_site)} wells declined; {n_recovering} recovered")
print(f"median change: {median_change:+.1f} ft")
print("Correct.")

Seventy-three of eighty, with a median of nearly fifty feet. That is not a
handful of overpumped wells; that is a basin.

---

## Part 5 — Long and wide

`levels` is **long**: one row per measurement, well identified by a column. Most
analysis wants that. Most *plotting* of many series at once wants **wide**: one
column per well, one row per date.

In [ ]:
wide = levels.pivot_table(index="date", columns="site_no",
                          values="depth_to_water_ft")

print(f"long: {levels.shape}  ->  wide: {wide.shape}")
wide.iloc[:4, :4]

Look at the shape: 4,731 dates by 80 wells is 378,480 cells, from 9,046
measurements. **The wide form is 98% empty**, because no two wells were
measured on the same days.

That is the trade. Wide is convenient and sparse; long is compact and awkward.
For irregular field data, stay long as long as you can.

`melt` goes back the other way:

In [ ]:
back_to_long = (
    wide.reset_index()
    .melt(id_vars="date", var_name="site_no", value_name="depth_to_water_ft")
    .dropna(subset=["depth_to_water_ft"])
)
print(f"{len(back_to_long):,} rows back from wide — matches the original {len(levels):,}?"
      f"  {len(back_to_long) == len(levels)}")

### YOUR TURN 5

Resample the wide table to **annual means per well**, then find how many wells
have a value in every one of the last ten years (2015–2024).

- `annual_wide` — the wide table resampled to year end, using the mean
- `complete_recent` — the number of wells with no missing annual value in 2015–2024

In [ ]:
# YOUR TURN
annual_wide = ...
complete_recent = ...

In [ ]:
# CHECK
assert annual_wide.shape[1] == 80, f"expected 80 wells, got {annual_wide.shape[1]}"
recent = annual_wide.loc["2015":"2024"]
assert complete_recent == int(recent.notna().all().sum()), \
    f"expected {int(recent.notna().all().sum())}, got {complete_recent}"
print(f"{annual_wide.shape[0]} years x {annual_wide.shape[1]} wells")
print(f"{complete_recent} wells measured in every year 2015-2024")
print("Correct.")

**Think about this before next week:** very few wells have a gap-free recent
record. Any figure showing "the basin water level over time" is really showing
whichever wells happened to be visited, and that set changes year to year.

Making that limitation visible — rather than hiding it behind a smooth line — is
a figure design problem, and figure design is Week 8.

---

## Before you leave

1. *Kernel → Restart Kernel and Run All Cells*
2. Fix anything that breaks
3. Save

## What's due

**HW 5 — Data retrieval and cleaning**, Wednesday 10/7 at 11:59pm.

## Next week

matplotlib properly: subplots, colour that survives being printed, annotation,
and maps.

## Stuck?

- `KeyError` on `flow.loc["2006"]` means the index isn't a `DatetimeIndex`.
  Check with `type(df.index)` — `parse_dates=` plus `set_index` is the fix.
- `resample` raises "Only valid with DatetimeIndex" for the same reason.
- Frequency strings changed in pandas 2.2: it is `ME`/`YE`, not `M`/`Y`. Older
  tutorials will use the old ones and you will get a `FutureWarning`.
- A rolling mean that is all `NaN` usually means the window is longer than the
  series.
- Office hours: Tuesdays 1:00–2:00pm, Harshbarger 322B.